In [0]:
from pyspark.sql.functions import *

In [0]:
%sql
truncate table delta.`/Volumes/pyspark_streaming_catalog/sink/streaming_sink_volume/dir9SEP/2/data`

In [0]:
%sql
-- BY DEFAULT - DELTA
create table pyspark_streaming_catalog.source.colors
(color string, eventtime timestamp)

In [0]:
# -- BY DEFAULT - DELTA
# create table pyspark_streaming_catalog.sink.colors_update_agg
# (start timestamp, end timestamp, color string, count int)
# using delta
# location "/Volumes/pyspark_streaming_catalog/sink/streaming_sink_volume/dir9SEP/2/data"

dfss = spark.createDataFrame([], """
start timestamp, end timestamp, color string, count long""")
dfss.write.format("delta").mode("overwrite").save("/Volumes/pyspark_streaming_catalog/sink/streaming_sink_volume/dir9SEP/2/data")

In [0]:
df = spark.readStream.table('pyspark_streaming_catalog.source.colors')\
           .withWatermark("eventtime", "30 minutes")
df.createOrReplaceTempView("df_readStream")

In [0]:
df = df.groupBy(window("eventtime", "5 minutes"), "color").agg(count("*").alias("count")).select("window.start", "window.end", "color", "count")

In [0]:
df.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "/Volumes/pyspark_streaming_catalog/sink/streaming_sink_volume/dir9SEP/1/checkpoint")\
        .trigger(once=True)\
    .start("/Volumes/pyspark_streaming_catalog/sink/streaming_sink_volume/dir9SEP/1/data")

In [0]:
from delta.tables import DeltaTable

delta_tbl_obj = DeltaTable.forPath(spark, '/Volumes/pyspark_streaming_catalog/sink/streaming_sink_volume/dir9SEP/2/data')

def foreachbatch_method(dfsrc, epochId):
    dfsrc.createOrReplaceTempView("dfsrc_foreachbatch")
    delta_tbl_obj.alias('trg').merge(
        dfsrc.alias('src'),
        'trg.start = src.start and trg.end = src.end and trg.color = src.color'
    ).whenMatchedUpdate(set = {"count":"trg.count+src.count"})\
        .whenNotMatchedInsertAll(condition="current_timestamp() > (src.end+INTERVAL '30 minutes')").execute()

df.writeStream.format("delta")\
    .outputMode("update")\
        .foreachBatch(foreachbatch_method)\
    .option("checkpointLocation", "/Volumes/pyspark_streaming_catalog/sink/streaming_sink_volume/dir9SEP/2/checkpoint")\
        .trigger(once=True)\
    .start()

In [0]:
df.writeStream.format("delta")\
    .outputMode("complete")\
    .option("checkpointLocation", "/Volumes/pyspark_streaming_catalog/sink/streaming_sink_volume/dir9SEP/3/checkpoint")\
        .trigger(once=True)\
    .start("/Volumes/pyspark_streaming_catalog/sink/streaming_sink_volume/dir9SEP/3/data")

In [0]:
df.writeStream.format("memory")\
    .queryName("update_memory")\
    .outputMode("update")\
        .option("checkpointLocation", "/Volumes/pyspark_streaming_catalog/sink/streaming_sink_volume/dir9SEP/4/checkpoint")\
        .trigger(once=True)\
    .start()

In [0]:
%sql
select * from delta.`/Volumes/pyspark_streaming_catalog/sink/streaming_sink_volume/dir9SEP/1/data`
order by start

In [0]:
%sql
select * from delta.`/Volumes/pyspark_streaming_catalog/sink/streaming_sink_volume/dir9SEP/2/data`
order by start

In [0]:
%sql
select * from delta.`/Volumes/pyspark_streaming_catalog/sink/streaming_sink_volume/dir9SEP/3/data`
order by start

In [0]:
%sql
describe history delta.`/Volumes/pyspark_streaming_catalog/sink/streaming_sink_volume/dir9SEP/1/data`

In [0]:
%sql
describe history delta.`/Volumes/pyspark_streaming_catalog/sink/streaming_sink_volume/dir9SEP/3/data`

In [0]:
%sql
select * from dfsrc_foreachbatch

In [0]:
df = spark.sql("select * from delta.`/Volumes/pyspark_streaming_catalog/sink/streaming_sink_volume/dir9SEP/1/data`")
df.printSchema()

In [0]:
%sql
select * from df_readStream

In [0]:
display(df, checkpointLocation = "/Volumes/pyspark_streaming_catalog/sink/streaming_sink_volume/dir9SEP/2/checkpoint")

In [0]:
%sql
select * from update_memory order by start